In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import pickle
from sOTDD.otdd.pytorch.sotdd import compute_pairwise_distance
from sOTDD.otdd.pytorch.datasets import load_torchvision_data
from sOTDD.otdd.pytorch.distance import DatasetDistance
from torch.utils.data.sampler import SubsetRandomSampler
import time
# from trainer import train, test_func, frozen_module
from sOTDD.models.resnet import *
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import argparse
from PIL import Image
import random
from sOTDD.wte.distance import WTE
from scipy.spatial import distance
from geoopt import Lorentz as Lorentz_geoopt
from sOTDD.hswfs_otdd.utils.hmds import HyperMDS
from sOTDD.hswfs_otdd.utils.bures_wasserstein import LabelsBW
from sOTDD.hswfs_otdd.hswfs.manifold.euclidean import Euclidean
from sOTDD.hswfs_otdd.hswfs.manifold.product import ProductManifold
from sOTDD.hswfs_otdd.hswfs.manifold.poincare import Poincare
from sOTDD.hswfs_otdd.hswfs.manifold.lorentz import Lorentz
from sOTDD.hswfs_otdd.hswfs.sw import sliced_wasserstein
from sliced_hierarchical_OT import compute_hierarchical_sliced_wasserstein

def calc_SHOT_between_subsets(sub1, sub2, length_scale=10., outer_pnum=100, grid_n=10, inner_pnum=100):
    data1_ls = []

    for label in sub1.classes:
        data1_ls.append(sub1.data[sub1.targets == label].reshape(-1, 28*28).float())
    
    data2_ls = []
    
    for label in sub2.classes:
        data2_ls.append(sub2.data[sub2.targets == label].reshape(-1, 28*28).float())
    return compute_hierarchical_sliced_wasserstein(data1_ls, data2_ls, length_scale=length_scale,
                                            outer_pnum=outer_pnum, grid_n=grid_n, inner_pnum=inner_pnum)

def generate_reference(num, dim_low, dim, attached_dim, seed=0):
    torch.manual_seed(seed)
    med = torch.rand(num, dim_low, dim_low).unsqueeze(0)
    s = dim/dim_low
    m = nn.Upsample(scale_factor=s, mode='bilinear')
    attached = torch.randn(num, attached_dim)
    return torch.cat((m(med).reshape(num, -1), attached), dim=1).float()



class Subset(Dataset):
    def __init__(self, dataset, original_indices, transform):
        self._dataset = dataset
        self._original_indices = original_indices
        self.transform = transform
        self.indices = torch.arange(start=0, end=len(self._original_indices), step=1)
        self.data = self._dataset.data[self._original_indices]
        self.targets = torch.tensor(self._dataset.targets)[self._original_indices]
        self.classes = sorted(torch.unique(torch.tensor(self._dataset.targets)).tolist())

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img = Image.fromarray(self.data[idx].numpy())
        return self.transform(img), self.targets[idx]
        # return self.data[idx], self.targets[idx]

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,))
])


def main():
    np.random.seed(42)
    torch.manual_seed(42)
    #parent_dir = "/homes/numerik/piening/scratch/InfinitedimensionalSlicing/sOTDD/s-OTDD"
    parent_dir = "/homes/numerik/piening/scratch/InfinitedimensionalSlicing/ConsolidatedCodeBase"
    parent_dir = f'{parent_dir}/correlation/MNIST'
    os.makedirs(parent_dir, exist_ok=True)

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # DEVICE = "cpu"
    print(f"Use CUDA or not: {DEVICE}")

    dataset = MNIST(root='data', train=True, download=True)
    test_dataset = MNIST(root='data', train=False, download=True, transform=transform)

    num_classes = len(torch.unique(dataset.targets))

    indices = np.arange(len(dataset))
    shuffled_indices = np.random.permutation(indices)
    
    max_dataset_size = len(dataset) // 20#2
    print(f"Maximum number of datapoint for each dataset: {max_dataset_size}")
    
    pointer_dataset1 = 0
    pointer_dataset2 = max_dataset_size

    list_dataset_size = [random.randint(5, 10) * 100 for i in range(100)]


    print(list_dataset_size)

    for idx in range(len(list_dataset_size)):
        dataset_size = list_dataset_size[idx]
        save_dir = f"{parent_dir}/seed_{idx}_size_{dataset_size}"
        os.makedirs(save_dir, exist_ok=True)

        shuffled_indices = np.random.permutation(indices)
        print(f"Setting dataset to size of {dataset_size}..")
        idx1 = shuffled_indices[:dataset_size]
        idx2 = shuffled_indices[-dataset_size:]

        print(f"len(idx1): {len(idx1)}, len(idx2): {len(idx2)}")

        sub1 = Subset(dataset=dataset, original_indices=idx1, transform=transform)
        sub2 = Subset(dataset=dataset, original_indices=idx2, transform=transform)

        subdatasets = [sub1, sub2]

        dataloader1 = DataLoader(sub1, batch_size=128, shuffle=True)
        dataloader2 = DataLoader(sub2, batch_size=128, shuffle=True)

        dataloaders = [dataloader1, dataloader2]


        # My sOTDD
        dict_new_sOTDD = torch.zeros(len(dataloaders), len(dataloaders))
        print("Compute Our OTDD")
        for i in range(len(dataloaders)):
            for j in range(i+1, len(dataloaders)):
                dist = calc_SHOT_between_subsets(dataloaders[i].dataset, dataloaders[j].dataset, 
                                     length_scale=.1, outer_pnum=1000, grid_n=10, inner_pnum=10)
                d = dist.item()
                dict_new_sOTDD[i][j] = d
                dict_new_sOTDD[j][i] = d
        torch.save(dict_new_sOTDD, f'{save_dir}/my_sotdd_dist.pt')


        # OTDD
        dict_OTDD = torch.zeros(len(dataloaders), len(dataloaders))
        print("Compute OTDD (exact)...")
        for i in range(len(dataloaders)):
            for j in range(i+1, len(dataloaders)):
                dist = DatasetDistance(dataloaders[i],
                                        dataloaders[j],
                                        inner_ot_method='exact',
                                        debiased_loss=True,
                                        p=2,
                                        entreg=1e-3,
                                        device=DEVICE, λ_x=torch.tensor(0.).to(DEVICE)) 
                # \lambda_x = 0 (non-zero becuase of OTDD code error)
                
                d = dist.distance(maxsamples=None).item()
                dict_OTDD[i][j] = d
                dict_OTDD[j][i] = d
        torch.save(dict_OTDD, f'{save_dir}/exact_otdd_dist.pt')


        # sOTDD
        try:
            projection_list = [10_000]
            for proj_id in projection_list:
                pairwise_dist = torch.zeros(len(dataloaders), len(dataloaders))
                print("Compute sOTDD...")
                print(f"Number of datasets: {len(dataloaders)}")
                kwargs = {
                    "dimension": 784,
                    "num_channels": 1,
                    "num_moments": 5,
                    "use_conv": False,
                    "precision": "float",
                    "p": 2,
                    "chunk": 1000
                }
                #, duration_periods
                list_pairwise_dist = compute_pairwise_distance(list_D=dataloaders, num_projections=proj_id, device=DEVICE, evaluate_time=True, **kwargs)
                t = 0
                for i in range(len(dataloaders)):
                    for j in range(i+1, len(dataloaders)):
                        pairwise_dist[i, j] = list_pairwise_dist[t]
                        pairwise_dist[j, i] = list_pairwise_dist[t]
                        t += 1
                torch.save(pairwise_dist, f'{save_dir}/sotdd_{proj_id}_dist.pt')
        except:
            print("sOTDD failed...")


if __name__ == "__main__":
    main()


[pyKeOps] Compiling nvrtc binder for python ... 
[KeOps] Warning : There were warnings or errors :
In file included from /homes/numerik/piening/.local/lib/python3.11/site-packages/pybind11/include/pybind11/detail/../attr.h:13:0,
                 from /homes/numerik/piening/.local/lib/python3.11/site-packages/pybind11/include/pybind11/detail/class.h:12,
                 from /homes/numerik/piening/.local/lib/python3.11/site-packages/pybind11/include/pybind11/pybind11.h:13,
                 from /store/frege/datastore/piening/.local/lib/python3.11/site-packages/pykeops/common/keops_io/pykeops_nvrtc.cpp:9:
/homes/numerik/piening/.local/lib/python3.11/site-packages/pybind11/include/pybind11/detail/../detail/common.h:266:10: fatal error: Python.h: No such file or directory
 #include <Python.h>
          ^~~~~~~~~~
compilation terminated.

OK


ot.gpu not found - coupling computation will be in cpu
/store/frege/datastore/piening/InfinitedimensionalSlicing/ConsolidatedCodeBase/sOTDD/otdd/pytorch/sotdd.py:16: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
ot.gpu not found - coupling computation will be in cpu


Use CUDA or not: cuda
Maximum number of datapoint for each dataset: 3000
[800, 800, 500, 900, 700, 1000, 500, 1000, 1000, 500, 500, 900, 500, 500, 500, 1000, 700, 700, 900, 1000, 1000, 600, 800, 600, 1000, 700, 600, 900, 1000, 900, 1000, 900, 900, 700, 800, 700, 900, 500, 900, 600, 500, 500, 900, 700, 900, 500, 700, 800, 500, 700, 700, 600, 800, 900, 900, 900, 500, 700, 800, 1000, 800, 800, 900, 600, 600, 500, 700, 700, 1000, 1000, 800, 500, 800, 700, 500, 500, 1000, 700, 700, 500, 1000, 800, 1000, 900, 600, 900, 700, 800, 900, 700, 500, 900, 500, 700, 500, 1000, 1000, 500, 800, 600]
Setting dataset to size of 800..
len(idx1): 800, len(idx2): 800
Compute Our OTDD


/tmp/ipykernel_23763/2883506825.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.targets = torch.tensor(self._dataset.targets)[self._original_indices]
/tmp/ipykernel_23763/2883506825.py:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.classes = sorted(torch.unique(torch.tensor(self._dataset.targets)).tolist())


Compute OTDD (exact)...


...

  0%|          | 0/45 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

Compute sOTDD...
Number of datasets: 2


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]